In [ ]:
%pip install -q ipylab

# Autostart

Autostart is a feature implemented using the [pluggy](https://pluggy.readthedocs.io/en/stable/index.html#pluggy) plugin system. The code associated with the entry point `ipylab_autostart` will be called (imported) when `ipylab` is activated. `ipylab` will activate when Jupyterlab is started (provided `ipylab` is installed and enabled). 

There are no limitations to what can be done. However, it is recommended to import on demand to minimise the time required to launch.  Some possibilities include:
* Create and register custom commands;
* Create launchers;
* Modify the appearance of Jupyterlab.

## Entry points

Add the following in your `pyproject.toml`

``` toml
[project.entry-points.ipylab_autostart]
myproject = "myproject.pluginmodule" # This line should point to the module you want to import (without the .py)
```

In `myproject.pluginmodule.py` write code that will be run once, say to register commands and launchers.

Example:

```python
# @ipylab_plugin.py

import asyncio

async def startup():
    import ipylab
     
    #Do everything to startup

pluginmodule = None # The module name here should be the same as above.
asyncio.create_task(startup())
```

## Example launching a small app

In [ ]:
# @my_module.autostart.py

import ipylab


async def create_app(cwd, path):
    # The code in this function is called in the new kernel.
    # Ensure imports are performed inside the function.
    import ipywidgets as ipw

    import ipylab

    panel = ipylab.Panel()
    panel.title.label = path
    panel.title.caption = panel.app.kernelId
    error_button = ipw.Button(
        description="Do an error",
        tooltip="An error dialog will pop up when this is clicked.\n"
        "The dialog demonstrates the use of the `on_frontend_error` plugin.",
    )
    error_button.on_click(lambda _: panel.app.commands.execute_method("execute", "Not a command"))
    panel.children = [
        ipw.HTML(f"<h3>{path}</h3> Welcome to my app.<br> kernel id: {panel.app.kernelId}<br>{cwd=}"),
        error_button,
    ]

    # Demonstrate usage of a plugin
    class IpylabPlugins:
        # Define plugins (see IpylabHookspec for available hooks)
        @ipylab.hookimpl
        def on_frontend_error(self, obj, error, content):  # noqa: ARG002
            panel.app.dialog.show_error_message("Error", str(error))

    # Register plugin for this kernel.
    ipylab.hookspecs.pm.register(IpylabPlugins())  # type: ignore
    return panel


async def start_my_app(cwd):
    path = await ipylab.app.dialog.get_text("Path for app")
    if not path:
        n = getattr(start_my_app, "callcount", 0)
        n += 1
        start_my_app.callcount = n
        path = f"my app {n}"
    return await ipylab.app.shell.add(create_app, cwd=cwd, path=path)


async def register_commands():
    cmd = await ipylab.app.commands.add(
        "Start my app",
        execute=start_my_app,
        label="Start",
        caption="Start my custom app",
        icon_class="jp-PythonIcon",
    )
    await cmd.add_launcher("Ipylab")


ipylab_plugin = None
ipylab.app.to_task(register_commands())

In [ ]:
t = ipylab.app.execute_command("launcher:create")